[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/router.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239412-lesson-5-router)

# Router

## 回顧

我們建立了一個以 `messages` 作為 state 的 graph，並搭配了一個綁定 tools 的 chat model。

我們看到這個 graph 可以：

* 回傳一個 tool call
* 回傳一段自然語言回覆

## 目標

我們可以把它想成一個 router：chat model 會根據使用者輸入，在「直接回覆」與「呼叫 tool」之間做路由。

這是 agent 的一個簡單範例 —— LLM 透過「呼叫 tool」或「直接回覆」來主導控制流程。

![Screenshot 2024-08-21 at 9.24.09 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac6543c3d4df239a4ed1_router1.png)

讓我們擴充這個 graph，讓它能處理兩種輸出！

為此，我們可以運用兩個想法：

(1) 加入一個會呼叫我們 tool 的 node。

(2) 加入一條 conditional edge，它會檢視 chat model 的輸出，然後路由到我們的 tool calling node；如果沒有 tool call，就直接結束。



In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [1]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools([multiply])

 我們使用[內建的 `ToolNode`](https://langchain-ai.github.io/langgraph/reference/agents/#langgraph.prebuilt.tool_node.ToolNode)，只要把我們的 tool 串列傳進去就能初始化它。
 
 我們使用[內建的 `tools_condition`](https://langchain-ai.github.io/langgraph/reference/agents/#langgraph.prebuilt.tool_node.tools_condition) 作為我們的 conditional edge。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# Node
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 建構 graph
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # 如果 assistant 最新的 message（結果）是 tool call -> tools_condition 會路由到 tools
    # 如果 assistant 最新的 message（結果）不是 tool call -> tools_condition 會路由到 END
    tools_condition,
)
builder.add_edge("tools", END)
graph = builder.compile()

# 檢視
display(Image(graph.get_graph().draw_mermaid_png()))

In [8]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="Hello, what is 2 multiplied by 2?")]
messages = graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hello world.
================================== Ai Message ==================================

Hello! How can I assist you today?


現在，我們可以看到 graph 真的執行了 tool！

它以一個 `ToolMessage` 回覆。

## Studio

**⚠️ 注意**

在拍攝這些影片之後，我們更新了 Studio，現在它可以在本機執行並透過你的瀏覽器存取。相較於影片中所示的桌面 App，這是執行 Studio 的建議方式。它現在改名為 _LangSmith Studio_，而非 _LangGraph Studio_。詳細的設定說明請見課程一開始的「Getting Setup」指南。你可以在[這裡](https://docs.langchain.com/langsmith/studio)找到 Studio 的說明，並在[這裡](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本機部署的具體細節。
若要啟動本機開發伺服器，請在你的終端機中、於本模組的 `/studio` 目錄下執行以下指令：

```
langgraph dev
```

你應該會看到以下輸出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

開啟你的瀏覽器，前往上面顯示的 Studio UI。
在 Studio 中載入 `router`，它使用的是 `module-1/studio/langgraph.json` 中所設定的 `module-1/studio/router.py`。

In [7]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")